# 06 - Embedding & hybrid models

Tests whether ESM-2 embeddings break the handcrafted-feature ceiling (macro-F1
~0.527 from notebook 04), and whether combining both representations beats either
one alone.

**Five rows of comparison** (the three new ones plus the two best baselines):

| Feature set | Model | from |
|---|---|---|
| Handcrafted (32) | XGBoost | NB04 baseline |
| ESM (1280) | LogReg | here |
| ESM (1280) | XGBoost | here |
| Handcrafted + ESM (1312) | XGBoost | here (hybrid) |

**Discipline:** identical proteins, identical train/test split as NB04 — only the
*features* change. That's what makes the comparison fair. All models run on CPU;
no GPU needed for this notebook.


## 1. Setup & load

In [ ]:
# Local: install if needed. (xgboost + scikit-learn + matplotlib)
# !pip install xgboost scikit-learn matplotlib seaborn


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EMB_DIR       = DATA_DIR / "embeddings"
RESULTS_DIR   = PROJECT_ROOT / "results"
SRC_DIR       = PROJECT_ROOT / "src"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))
from embeddings import load_embeddings

# Class order must match notebook 04 so confusion matrices / reports line up.
CLASS_ORDER = ["enzyme", "dna_rna_binding", "receptor",
               "transporter", "structural", "other"]
RANDOM_STATE = 42


## 2. Load the three data sources and align them

Three files, all keyed by `accession`:
- `labeled_dataset.csv` — the labels (and sequences, used only for dedup)
- `sequence_features.csv` — the 32 handcrafted features (NB03 output)
- `esm_embeddings.npy` + `metadata.csv` — the embeddings (NB05 output)

We align everything to one shared set of accessions so row `i` means the same
protein in every matrix.

In [ ]:
# Labels + sequences
labels_df = pd.read_csv(PROCESSED_DIR / "labeled_dataset.csv")

# Handcrafted features (indexed by accession, as saved in NB03)
feats = pd.read_csv(PROCESSED_DIR / "sequence_features.csv", index_col="accession")

# Embeddings + their accession order
emb, emb_meta = load_embeddings(EMB_DIR)
emb_df = pd.DataFrame(emb, index=emb_meta["accession"].astype(str))
emb_df.index.name = "accession"

print(f"labels:      {labels_df.shape}")
print(f"handcrafted: {feats.shape}")
print(f"embeddings:  {emb_df.shape}")


### Drop duplicate sequences (leakage control), then take the shared accessions

Same dedup as NB04/NB05. After dedup we intersect the accessions present in all
three sources, so every model trains and tests on exactly the same proteins.

In [ ]:
# Dedup by sequence (keep first), matching the earlier notebooks.
before = len(labels_df)
labels_df = labels_df.drop_duplicates(subset=["sequence"]).reset_index(drop=True)
print(f"Dropped {before - len(labels_df):,} duplicate sequences")

labels_df["accession"] = labels_df["accession"].astype(str)
feats.index = feats.index.astype(str)

# Accessions present in all three sources, in label order.
shared = (
    labels_df.set_index("accession")
    .index.intersection(feats.index)
    .intersection(emb_df.index)
)
labels_df = labels_df[labels_df["accession"].isin(shared)].reset_index(drop=True)
order = labels_df["accession"].tolist()   # canonical row order

y = labels_df["function_class"].to_numpy()
X_hand = feats.loc[order].to_numpy()
X_emb  = emb_df.loc[order].to_numpy()

# Hybrid = handcrafted (scaled later) concatenated with embeddings.
n_hand = X_hand.shape[1]
X_hybrid = np.hstack([X_hand, X_emb])

print(f"Aligned proteins: {len(order):,}")
print(f"  handcrafted dims: {X_hand.shape[1]}")
print(f"  embedding   dims: {X_emb.shape[1]}")
print(f"  hybrid      dims: {X_hybrid.shape[1]}")


## 3. Stratified train/test split

One split, reused for all feature sets via row indices — so handcrafted, ESM, and
hybrid models all see the identical train and test proteins.

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(order))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

y_train, y_test = y[train_idx], y[test_idx]
print(f"Train: {len(train_idx):,}   Test: {len(test_idx):,}")


## 4. Define the models

- **ESM + LogReg** — embeddings are dense and well-scaled, but we still standardize
  for LogReg's sake; `max_iter` raised for convergence on 1280 dims.
- **ESM + XGBoost** — no scaling needed (trees are scale-invariant).
- **Hybrid + XGBoost** — embeddings need no scaling and trees don't require it for
  the handcrafted block either, so a plain concatenation is fine for XGBoost. (If
  you later add a *linear* hybrid model, scale only the handcrafted columns with a
  ColumnTransformer.)

We use a small helper to fit, predict, and score each model the same way, and to
mirror NB04's metric set (CV macro-F1 on train + test macro/weighted-F1).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier

# XGBoost needs integer labels; encode once, decode for reports.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(CLASS_ORDER)

def make_xgb():
    return XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        objective="multi:softprob", num_class=len(CLASS_ORDER),
        eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1,
    )

def make_logreg():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE),
    )

def evaluate(name, model, X, needs_int_labels):
    """5-fold CV macro-F1 on train, then fit on train and score on test."""
    Xtr, Xte = X[train_idx], X[test_idx]
    if needs_int_labels:
        ytr, yte = le.transform(y_train), le.transform(y_test)
        scoring_y = ytr
    else:
        ytr, yte = y_train, y_test
        scoring_y = ytr

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_f1 = cross_val_score(model, Xtr, scoring_y, cv=cv,
                            scoring="f1_macro", n_jobs=-1)

    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    if needs_int_labels:
        pred_lbl, true_lbl = le.inverse_transform(pred), y_test
    else:
        pred_lbl, true_lbl = pred, y_test

    return {
        "feature_set": name.split(" | ")[0],
        "model": name.split(" | ")[1],
        "cv_macro_f1_mean": round(cv_f1.mean(), 4),
        "cv_macro_f1_std": round(cv_f1.std(), 4),
        "test_accuracy": round(accuracy_score(true_lbl, pred_lbl), 4),
        "test_macro_f1": round(f1_score(true_lbl, pred_lbl, average="macro"), 4),
        "test_weighted_f1": round(f1_score(true_lbl, pred_lbl, average="weighted"), 4),
    }, (true_lbl, pred_lbl)


## 5. Run the new models

In [ ]:
rows = []
preds = {}

r, p = evaluate("ESM (1280) | logreg",  make_logreg(), X_emb,    needs_int_labels=False)
rows.append(r); preds["ESM | logreg"] = p

r, p = evaluate("ESM (1280) | xgboost", make_xgb(),    X_emb,    needs_int_labels=True)
rows.append(r); preds["ESM | xgboost"] = p

r, p = evaluate("Hybrid (1312) | xgboost", make_xgb(), X_hybrid, needs_int_labels=True)
rows.append(r); preds["Hybrid | xgboost"] = p

new_df = pd.DataFrame(rows)
new_df


## 6. Combine with the Day 5 baselines

Pull in `results/baseline_metrics.csv` (NB04) so the full comparison lives in one
table. We keep the two strongest baselines for the headline plot.

In [ ]:
base = pd.read_csv(RESULTS_DIR / "baseline_metrics.csv")
# NB04 table has a `model` column; tag its feature set for the merge.
base = base.assign(feature_set="Handcrafted (32)")
base = base.rename(columns={"model": "model"})

# Keep the common columns and stack.
cols = ["feature_set", "model", "cv_macro_f1_mean", "cv_macro_f1_std",
        "test_accuracy", "test_macro_f1", "test_weighted_f1"]
combined = pd.concat([base[cols], new_df[cols]], ignore_index=True)
combined = combined.sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
combined


## 7. Comparison plot

In [ ]:
plot_df = combined.copy()
plot_df["label"] = plot_df["feature_set"] + "\n" + plot_df["model"]
plot_df = plot_df.sort_values("test_macro_f1")

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(plot_df["label"], plot_df["test_macro_f1"])
ax.axvline(0.527, ls="--", lw=1, color="grey")
ax.text(0.527, -0.6, " NB04 ceiling 0.527", color="grey", fontsize=9, va="top")
ax.set_xlabel("Test macro-F1")
ax.set_title("Protein function classification: feature set x model")
for b, v in zip(bars, plot_df["test_macro_f1"]):
    ax.text(v + 0.005, b.get_y() + b.get_height()/2, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "figures" / "model_comparison_barplot.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Detailed report + confusion matrix for the best model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

best_key = combined.iloc[0]["feature_set"].split()[0]
# Map the winning row to one of the prediction sets we stored.
key_lookup = {
    ("ESM (1280)", "logreg"): "ESM | logreg",
    ("ESM (1280)", "xgboost"): "ESM | xgboost",
    ("Hybrid (1312)", "xgboost"): "Hybrid | xgboost",
}
top = combined.iloc[0]
pred_key = key_lookup.get((top["feature_set"], top["model"]))

if pred_key:   # winner is one of the new models
    true_lbl, pred_lbl = preds[pred_key]
    print(f"Best model: {top['feature_set']} | {top['model']}  "
          f"(test macro-F1 {top['test_macro_f1']})\n")
    print(classification_report(true_lbl, pred_lbl,
                                labels=CLASS_ORDER, target_names=CLASS_ORDER,
                                zero_division=0))

    cm = confusion_matrix(true_lbl, pred_lbl, labels=CLASS_ORDER)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_ORDER))); ax.set_xticklabels(CLASS_ORDER, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_ORDER))); ax.set_yticklabels(CLASS_ORDER)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix - {top['feature_set']} {top['model']}")
    for i in range(len(CLASS_ORDER)):
        for j in range(len(CLASS_ORDER)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "figures" / "confusion_matrix_best.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Best model is a Day 5 baseline; see notebook 04 for its report.")


## 9. Save the combined comparison

In [ ]:
out_path = RESULTS_DIR / "model_comparison.csv"
combined.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")
combined


## 10. Interpretation

_Fill in after running._ The questions this notebook answers:

- **Did ESM break the 0.527 ceiling?** Compare ESM/XGBoost vs the best handcrafted
  model. A clear jump confirms the bottleneck was representation, as predicted.
- **Linear vs non-linear on embeddings.** ESM/LogReg vs ESM/XGBoost: if LogReg is
  now competitive, it means ESM embeddings are *linearly* separable in a way the 32
  handcrafted features were not — a meaningful finding about the representation.
- **Does hybrid help?** Hybrid/XGBoost vs ESM/XGBoost: if hybrid barely moves, the
  embeddings already capture what the handcrafted features encode (likely). If it
  lifts the weaker classes (receptor, structural), the handcrafted biochemistry adds
  complementary signal.
- **Per-class shifts.** Compare the confusion matrix to NB04's. Which classes did
  ESM rescue? `receptor` (hardest at F1 0.38) is the one to watch.

Keep claims hedged ("suggests", "is consistent with") — this feeds the Step 3
interpretability writeup and the Galaxy validation.
